# 04: Self-Supervised Learning (SSL) Sequence Embeddings

**Objective:** Extract dense vectorial representations (embeddings) representing physician temporal behavior.

**Key Actions:**
- Design skeletal 1D-CNN with Global Average Pooling.
- Inference loop to acquire 1D dense embeddings per physician (for all 20,930 IDs).
- Export embeddings matrix.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

# tensor_3d = np.load('data/tensors/longitudinal_tensor.npy')
# unique_ids = np.load('data/tensors/tensor_ids.npy')

## 1. 1D-CNN Architecture Skeleton

In [ ]:
class SequenceEmbedding1DCNN(nn.Module):
    """
    Lightweight 1D-CNN to extract sequence embeddings over 86 weeks.
    """
    def __init__(self, num_features: int, embedding_dim: int = 64):
        super(SequenceEmbedding1DCNN, self).__init__()
        self.conv_block = nn.Sequential(
            nn.Conv1d(in_channels=num_features, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=32, out_channels=embedding_dim, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
    def forward(self, x):
        # x shape: (Batch, Seq_Len, Features) -> (Batch, Features, Seq_Len) for Conv1D
        x = x.transpose(1, 2)
        x = self.conv_block(x)
        # x shape after conv: (Batch, embedding_dim, Seq_Len)
        x = self.global_pool(x)
        # x shape after pool: (Batch, embedding_dim, 1)
        return x.squeeze(-1) # (Batch, embedding_dim)

def initialize_model(num_features: int) -> nn.Module:
    logger.info("Initializing 1D-CNN Embedding Model Architecture...")
    return SequenceEmbedding1DCNN(num_features=num_features)

## 2. Unsupervised Embedding Inference
Extract vectors directly. (In a full flow, this model would first be trained via TS2Vec/SimCLR contrastive objectives).

In [ ]:
def extract_embeddings(tensor_array: np.ndarray, model: nn.Module, batch_size: int = 1024) -> np.ndarray:
    """
    Batched inference to extract embeddings bridging the OOM issues for large matrices.
    """
    logger.info("Extracting global sequence embeddings via inference loop...")
    model.eval()
    all_embeddings = []
    
    if len(tensor_array) == 0: return np.array([])
    
    with torch.no_grad():
        for start_idx in range(0, len(tensor_array), batch_size):
            end_idx = start_idx + batch_size
            batch_tensor = torch.tensor(tensor_array[start_idx:end_idx], dtype=torch.float32)
            
            embed_batch = model(batch_tensor)
            all_embeddings.append(embed_batch.numpy())
            
    embeddings_matrix = np.vstack(all_embeddings)
    logger.info(f"Embeddings extraction complete. Final shape: {embeddings_matrix.shape}")
    return embeddings_matrix

# num_features_input = tensor_3d.shape[2] if 'tensor_3d' in locals() else 1
# dummy_model = initialize_model(num_features=num_features_input)
## embeddings_matrix = extract_embeddings(tensor_3d, dummy_model)

## emb_cols = [f'emb_{i}' for i in range(embeddings_matrix.shape[1])]
## embeddings_df = pd.DataFrame(embeddings_matrix, columns=emb_cols)
## embeddings_df['NUEVO_ID'] = unique_ids
## embeddings_df.to_parquet('data/features/sequence_embeddings.parquet', index=False)